In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = "/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data"
OUTPUT_PATH = os.path.join(BASE_PATH, "phase3_outputs")

In [2]:
q30_malaria = pd.read_csv(f"{BASE_PATH}/phase3_outputs/q30_wash_malaria.csv")
q30_tb = pd.read_csv(f"{BASE_PATH}/phase3_outputs/q30_wash_tb.csv")

for name, df in [("malaria", q30_malaria), ("tb", q30_tb)]:
    print(f"--- {name} ---")
    print("shape:", df.shape, "| countries:", df["country"].nunique(),
          "| years:", df["year"].min(), "-", df["year"].max())
    print(df.isna().sum())
    print()

--- malaria ---
shape: (1900, 5) | countries: 108 | years: 2000 - 2017
country           0
year              0
water_pct         0
sanitation_pct    0
malaria_inc       0
dtype: int64

--- tb ---
shape: (3426, 5) | countries: 195 | years: 2000 - 2017
country           0
year              0
water_pct         0
sanitation_pct    0
tb_inc            0
dtype: int64



In [3]:
# Consistent with prior convention: period averages over the full window,
# since coverage is consistent within each joined file.
malaria_avg = q30_malaria.groupby("country")[["water_pct", "sanitation_pct", "malaria_inc"]].mean().reset_index()
tb_avg = q30_tb.groupby("country")[["water_pct", "sanitation_pct", "tb_inc"]].mean().reset_index()

print("Malaria countries:", len(malaria_avg))
print("TB countries:", len(tb_avg))
malaria_avg.describe()

Malaria countries: 108
TB countries: 195


,water_pct,sanitation_pct,malaria_inc
count,108.000000,108.000000,108.000000
mean,64.025374,54.198104,109.534439
std,22.928073,29.753621,151.498362
min,19.291111,5.363333,0.000000
25%,44.746944,26.811389,1.505156
50%,65.666193,52.881944,16.419951
75%,84.078889,81.064722,204.598611
max,99.015625,100.000000,510.838889


In [ ]:
from scipy.stats import spearmanr

combos = [
    ("water_pct", "malaria_inc", malaria_avg),
    ("sanitation_pct", "malaria_inc", malaria_avg),
    ("water_pct", "tb_inc", tb_avg),
    ("sanitation_pct", "tb_inc", tb_avg),
]

for wash_col, disease_col, df in combos:
    rho, p = spearmanr(df[wash_col], df[disease_col])
    print(f"{wash_col} vs {disease_col}: rho={rho:.3f}, p={p:.4f}, n={len(df)}")